# Retail promotion measurement

A retailer can send one of two merchandise emails or hold a customer out. How many additional customers buy within two weeks for every 1,000 assigned to each email? The historical randomized experiment records one row per customer. The terms *men’s* and *women’s* identify the merchandise featured in the creative.

## 1. Load the verified experiment

Use the same ingestion code as the command-line analysis. The loader checks the source fingerprint and the number of customers assigned to each arm. An in-memory SQLite database keeps this walkthrough self-contained.

In [1]:
from pathlib import Path
import sqlite3
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "run_analysis.py").is_file())
sys.path.insert(0, str(root))

from promotion.analysis import contrasts, summarize
from promotion.data import CONTROL, MENS, load_assignments, read_source
from promotion.report import decision

db = sqlite3.connect(":memory:")
load_assignments(db, read_source())
print(f"{db.execute('SELECT COUNT(*) FROM assignments').fetchone()[0]:,} assigned customers")

64,000 assigned customers


## 2. Count the buyers in SQL

The denominator includes everyone assigned to the email or holdout. Restricting it to visitors would change the question because the email itself can affect whether a customer visits.

In [2]:
print((root / "sql/arm_summary.sql").read_text())

-- One record per assigned customer. Non-visitors stay in all denominators.
SELECT
    segment AS arm,
    COUNT(*) AS assigned,
    SUM(visit) AS visitors,
    SUM(conversion) AS buyers,
    AVG(1.0 * conversion) AS purchase_rate,
    SUM(spend_cents) AS total_spend_cents,
    SUM(spend_cents) / (100.0 * COUNT(*)) AS spend_per_assigned
FROM assignments
GROUP BY segment
ORDER BY CASE segment
    WHEN 'No E-Mail' THEN 1
    WHEN 'Mens E-Mail' THEN 2
    WHEN 'Womens E-Mail' THEN 3
END;



In [3]:
arms = summarize(db)
print(f"{'Assignment':<18}{'Assigned':>10}{'Visitors':>10}{'Buyers':>8}{'Purchase rate':>16}")
for a in arms:
    print(f"{a.arm:<18}{a.assigned:>10,}{a.visitors:>10,}{a.buyers:>8,}{a.purchase_rate:>16.3%}")

Assignment          Assigned  Visitors  Buyers   Purchase rate
No E-Mail             21,306     2,262     122          0.573%
Mens E-Mail           21,307     3,894     267          1.253%
Womens E-Mail         21,387     3,238     189          0.884%


## 3. Estimate additional buyers per 1,000

Subtract the holdout purchase rate from each email purchase rate, then multiply by 1,000. We also compare the creatives directly because the business can choose only one email. Newcombe score intervals quantify uncertainty in each risk difference. A Bonferroni adjustment covers the family of three comparisons at approximately 95% confidence.

In [4]:
effects = contrasts(arms)
print(f"{'Comparison':<30}{'Extra buyers / 1,000':>22}{'Adjusted interval':>23}")
for effect in effects:
    pair = f"{effect.treatment} vs {effect.comparison}"
    interval = f"[{effect.ci_low:+.2f}, {effect.ci_high:+.2f}]"
    print(f"{pair:<30}{effect.extra_buyers_per_1000:>+22.2f}{interval:>23}")

# Verify the first result directly against the SQL counts.
by_name = {a.arm: a for a in arms}
email, holdout = by_name[MENS], by_name[CONTROL]
manual = 1000 * (email.buyers / email.assigned - holdout.buyers / holdout.assigned)
assert abs(manual - effects[0].extra_buyers_per_1000) < 1e-10

Comparison                      Extra buyers / 1,000      Adjusted interval
Mens E-Mail vs No E-Mail                       +6.81         [+4.62, +9.06]
Womens E-Mail vs No E-Mail                     +3.11         [+1.14, +5.12]
Mens E-Mail vs Womens E-Mail                   +3.69         [+1.31, +6.11]


## 4. Make the send decision

In [5]:
print(decision(effects))
db.close()

Choose the men’s-merchandise email to maximize the number of buyers in a comparable audience.


The men’s creative generated an estimated **6.81 extra buyers per 1,000** versus holdout, and **3.69 extra buyers per 1,000** versus the women’s creative. The adjusted head-to-head interval runs from **1.31 to 6.11**. This recommendation applies to two-week purchase conversion among customers comparable to the 2008 challenge audience. A profit decision would also require margins and campaign costs.

![Purchase lift and adjusted confidence intervals](../reports/campaign_lift.png)

[Campaign decision memo](../reports/decision.md) · [Data provenance and dictionary](../data/README.md)